# v7 — RoBERTa-base 3-class, int8 + prior correction
No new training: the v1 model is quantized to int8 (249 → ~126 MB) and scored on the balanced test,
the natural class mix, and the natural mix with the prior correction (logits + log(class prior)).
int8 runs on CPU only (the T4 scores the fp32 reference). Runtime → T4 GPU → Run all. ~25 min
(~70 min if the v1 model is not on Drive and has to be retrained).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/btp_v7_int8_3class_prior'
    V1_ZIP = '/content/drive/MyDrive/btp_v1_roberta_base_3class/btp_v1_roberta_base_3class.zip'
except Exception as e:
    print(f'Drive mount failed ({e}); the v1 model will be retrained.')
    DRIVE, V1_ZIP = '/content/btp_v7_int8_3class_prior', ''
!mkdir -p {DRIVE}
!test -d /content/BTP || git clone -q https://github.com/Abhijeet-SP/BTP.git /content/BTP
%cd /content/BTP
!git fetch -q origin && git reset -q --hard origin/main && git log --oneline -1
RESULTS = 'versions/v7_int8_3class_prior/results'
!pip -q install -U transformers datasets accelerate scikit-learn

In [ ]:
import os
# Re-set here: a Colab restart (e.g. after the pip upgrade) wipes variables and the working dir
%cd /content/BTP
RESULTS = 'versions/v7_int8_3class_prior/results'
DRIVE = '/content/drive/MyDrive/btp_v7_int8_3class_prior' if os.path.exists('/content/drive/MyDrive') else '/content/btp_v7_int8_3class_prior'
V1_ZIP = '/content/drive/MyDrive/btp_v1_roberta_base_3class/btp_v1_roberta_base_3class.zip'
# ~8 min: same balanced test.csv + test_natural.csv as v1 (seeded)
!python prepare_data.py

In [ ]:
import os
# Re-set here: a Colab restart (e.g. after the pip upgrade) wipes variables and the working dir
%cd /content/BTP
RESULTS = 'versions/v7_int8_3class_prior/results'
DRIVE = '/content/drive/MyDrive/btp_v7_int8_3class_prior' if os.path.exists('/content/drive/MyDrive') else '/content/btp_v7_int8_3class_prior'
V1_ZIP = '/content/drive/MyDrive/btp_v1_roberta_base_3class/btp_v1_roberta_base_3class.zip'
# The v1 model: from the zip the v1 notebook left on Drive, else retrain it with the v1 recipe (~42 min)
if not os.path.exists('models/roberta_base_3class/config.json'):
    if V1_ZIP and os.path.exists(V1_ZIP):
        !unzip -qo "{V1_ZIP}" 'models/roberta_base_3class/*'
    else:
        print('v1 zip not on Drive, retraining v1')
        !python finetune_roberta.py --name roberta_base_3class --batch-size 32 --results-dir {RESULTS} --ckpt-dir {DRIVE}/ckpt
assert os.path.exists('models/roberta_base_3class/config.json'), 'No v1 model: scroll up. Re-run the cell to resume.'
print('v1 model ready')

In [ ]:
import os
# Re-set here: a Colab restart (e.g. after the pip upgrade) wipes variables and the working dir
%cd /content/BTP
RESULTS = 'versions/v7_int8_3class_prior/results'
DRIVE = '/content/drive/MyDrive/btp_v7_int8_3class_prior' if os.path.exists('/content/drive/MyDrive') else '/content/btp_v7_int8_3class_prior'
V1_ZIP = '/content/drive/MyDrive/btp_v1_roberta_base_3class/btp_v1_roberta_base_3class.zip'
# fp32 on the T4, int8 on the CPU; same 4,000 rows of each test set
!python evaluate_model.py --model models/roberta_base_3class --quantize --results-dir {RESULTS} 2>&1 | tee /tmp/quantize.log
!mkdir -p {RESULTS}/logs && cp /tmp/quantize.log {RESULTS}/logs/quantize.log

In [ ]:
import os
# Re-set here: a Colab restart (e.g. after the pip upgrade) wipes variables and the working dir
%cd /content/BTP
RESULTS = 'versions/v7_int8_3class_prior/results'
DRIVE = '/content/drive/MyDrive/btp_v7_int8_3class_prior' if os.path.exists('/content/drive/MyDrive') else '/content/btp_v7_int8_3class_prior'
V1_ZIP = '/content/drive/MyDrive/btp_v1_roberta_base_3class/btp_v1_roberta_base_3class.zip'
import json
m = json.load(open(f'{RESULTS}/metrics.json'))
for t in ('fp32', 'int8'):
    for s in ('', '_natural', '_natural_prior'):
        k = f'roberta_base_3class_{t}{s}'
        v = m[k]
        print(f"{k:38s} acc {v['accuracy']:.4f}  macro-F1 {v['macro_f1']:.4f}")
    print(f"   size {m[f'roberta_base_3class_{t}']['model_size_mb']} MB, "
          f"{m[f'roberta_base_3class_{t}']['reviews_per_second']} reviews/s on {m[f'roberta_base_3class_{t}']['device']}")

In [ ]:
import os
# Re-set here: a Colab restart (e.g. after the pip upgrade) wipes variables and the working dir
%cd /content/BTP
RESULTS = 'versions/v7_int8_3class_prior/results'
DRIVE = '/content/drive/MyDrive/btp_v7_int8_3class_prior' if os.path.exists('/content/drive/MyDrive') else '/content/btp_v7_int8_3class_prior'
V1_ZIP = '/content/drive/MyDrive/btp_v1_roberta_base_3class/btp_v1_roberta_base_3class.zip'
!zip -qr btp_v7_int8_3class_prior.zip models/roberta_base_3class_int8.pt versions/v7_int8_3class_prior/results && ls -lh btp_v7_int8_3class_prior.zip
!cp btp_v7_int8_3class_prior.zip {DRIVE}/
from google.colab import files
files.download('btp_v7_int8_3class_prior.zip')